# **Dr Shaheen Khatoon **

Ph.D. | PMP | PMI-ACP | ITIL | PgCert TLHE

Senior Lecturer in Data Science (Computer Science and Digital Technologies)

School of Architecture, Computing and Engineering

University of East London, UK

# Learning Outcomes
**In this tutotial you are going to learn:**

  1: Basic NLP techniques to preprocess text

  2: Differnt techniques to conver text into numerical form (CountVectorize, HashingTF, and TF-IDF)

  3: Apply regression classifier on the text corpus

**Install spark and otehr necessary preliminiries**

In [ ]:
!pip install pyspark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 1.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pyspark: filename=pyspark-3.5.1-py2.py3-none-any.whl size=317488493 sha256=bd5bf95014b1ca9e33c60b73700e74535b31d9004e134ae77949c46b9bffa9ee
  Stored in directory: /root/.cache/pip/wheels/80/1d/60/2c256ed38dddce2fdd93be545214a63e02fbd8d74fb0b7f3a6
Successfully built pyspark


In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()

In [ ]:
spark = SparkSession \
    .builder \
    .appName("Spark_NLP") \
    .config("spark.some.config.option", "some-value") \
    .getOrCreate()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#**Steps involve in NLP:**
Text data can be very messy sometimes, and it needs careful attention to bring it to a stage where it can be used in the right way. There are multiple ways in which text data can be cleaned and refined. For example, regular expressions are very powerful when it comes to filtering, cleaning, and standardizing text data. However, regular expressions are not the focus area in this tutorial. Rather, we look at the steps to prepare text data in a form where we can fit a ML model on it. The five major steps involved in handling text data for ML modeling are:

1.   Reading the corpus

2.	Tokenization

3.	Cleaning/stopword removal

4.	Stemming

5.	Converting into numerical form


**Creating Carpous **

A corpus is known as an entire collection of text documents, for example, a collection of emails, messages, or user reviews. This group of individual text items is known as a **corpus**.
We will use CSV file contain collection of text later in this tutorial. First we will do some basic pre-processing using text.

##**1. Tokenization**

The method of dividing the given text sequence/sentence or collection of words of a text document into individual words is known as tokenization. It removes the unnecessary characters such as punctuations. The final units post-tokenization are known as tokens. Let’s say we have the following text:

**Input:** 'He really liked the London City. He is there for two more days'

Tokenization would result in the following tokens.

[He, really, liked, the, London, City, He, is, there, for, two, more, days]

We end up with 13 tokens for the input text:

Let us see how we can do tokenization using PySpark.
The first step is to create a dataframe that has text data.

In [ ]:
#create data frame

df2=spark.createDataFrame([(1,'I really liked this movie wathing this movie was an unforgetable experience'),
                         (2,'I would recommend this movie to my friends'),
                         (3,'movie was alright but acting was horrible'),
                         (4,'I am never watching that movie ever again')],
                        ['user_id','review'])
df2.show(4,False)
# When False is passed as the second argument, it means that long strings will not be truncated.

+-------+---------------------------------------------------------------------------+
|user_id|review                                                                     |
+-------+---------------------------------------------------------------------------+
|1      |I really liked this movie wathing this movie was an unforgetable experience|
|2      |I would recommend this movie to my friends                                 |
|3      |movie was alright but acting was horrible                                  |
|4      |I am never watching that movie ever again                                  |
+-------+---------------------------------------------------------------------------+



In this Dataframe, we have four sentences for tokenization. The next step is to import Tokenizer from the **Spark library**. We must then pass the input column and name the output column after tokenization. We use the transform function to apply tokenization to the review column:

In [ ]:
from pyspark.ml.feature import Tokenizer
tokenization=Tokenizer(inputCol='review',outputCol='tokens')
tokenized_df2=tokenization.transform(df2)
tokenized_df2.show(4,False)

+-------+---------------------------------------------------------------------------+----------------------------------------------------------------------------------------+
|user_id|review                                                                     |tokens                                                                                  |
+-------+---------------------------------------------------------------------------+----------------------------------------------------------------------------------------+
|1      |I really liked this movie wathing this movie was an unforgetable experience|[i, really, liked, this, movie, wathing, this, movie, was, an, unforgetable, experience]|
|2      |I would recommend this movie to my friends                                 |[i, would, recommend, this, movie, to, my, friends]                                     |
|3      |movie was alright but acting was horrible                                  |[movie, was, alright, but, acting, was, 

We get a new column named tokens that contains the tokens for each sentence.

## **2. Stopword Removal**

As you can observe, the tokens column contains very common words such as “this,” “the,” “to,” “was,” “that,” etc. These words are known as stopwords, and they seem to add very little value to the analysis. If they are to be used in analysis, it increases the computation overheads without adding any value or insight. Hence, it’s preferred to drop these stopwords from the tokens. In PySpark, we use **StopWordsRemover** to remove the stopwords:

In [ ]:
from pyspark.ml.feature import StopWordsRemover
stopword_removal=StopWordsRemover(inputCol='tokens',outputCol='refined_tokens')
#We then pass the tokens as the input column and name the output column as refined tokens.
refined_df2=stopword_removal.transform(tokenized_df2)
refined_df2.select(['user_id','tokens','refined_tokens']).show(4,False)

+-------+----------------------------------------------------------------------------------------+----------------------------------------------------------------+
|user_id|tokens                                                                                  |refined_tokens                                                  |
+-------+----------------------------------------------------------------------------------------+----------------------------------------------------------------+
|1      |[i, really, liked, this, movie, wathing, this, movie, was, an, unforgetable, experience]|[really, liked, movie, wathing, movie, unforgetable, experience]|
|2      |[i, would, recommend, this, movie, to, my, friends]                                     |[recommend, movie, friends]                                     |
|3      |[movie, was, alright, but, acting, was, horrible]                                       |[movie, alright, acting, horrible]                              |
|4      |[i, am,

##**3. Stemming/Lemmatization**

Lemmatization is a natural language processing technique that aims to reduce words to their base or dictionary form, known as the lemma. The base form is often referred to as the **root word or the canonical form**. Lemmatization takes into account the context and part of speech (POS) of a word in order to produce the most appropriate base form.

The main goal of lemmatization is to normalize words so that different inflected forms of a word are treated as a single item. For example, the lemma of "running" would be "run," and the lemma of "played" would be "play." By reducing words to their base forms, lemmatization helps to eliminate redundancy and improve the accuracy and efficiency of text analysis tasks.

Lemmatization is different from stemming, another popular text normalization technique. While stemming simply chops off prefixes or suffixes from words to derive the root form, lemmatization considers the vocabulary and morphological analysis of words to determine the appropriate base form.

In natural language processing workflows, lemmatization is often used in tasks such as information retrieval, text classification, topic modeling, and sentiment analysis. It helps to consolidate the representation of words, enabling better understanding and analysis of text data.








In [ ]:
#We will use nltk library to define and use lemmatizer.
import nltk
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [ ]:
from pyspark.sql.functions import udf
from pyspark.sql.types import ArrayType, StringType
from nltk.stem import WordNetLemmatizer

# Define lemmatization function using nltk library
def lemmatize_text(tokens):
    lemmatizer = WordNetLemmatizer()
    lemmas = [lemmatizer.lemmatize(token) for token in tokens]
    return lemmas

# Register lemmatization function as a UDF
lemmatize_udf = udf(lemmatize_text, ArrayType(StringType()))

# Apply lemmatization using the UDF
refined_df2 = refined_df2.withColumn("lemmatized_words", lemmatize_udf("refined_tokens"))

# Show the resulting DataFrame
refined_df2.show(truncate=False)


+-------+---------------------------------------------------------------------------+----------------------------------------------------------------------------------------+----------------------------------------------------------------+----------------------------------------------------------------+
|user_id|review                                                                     |tokens                                                                                  |refined_tokens                                                  |lemmatized_words                                                |
+-------+---------------------------------------------------------------------------+----------------------------------------------------------------------------------------+----------------------------------------------------------------+----------------------------------------------------------------+
|1      |I really liked this movie wathing this movie was an unforgetable experience|

Note for the second review 'friends' became 'friend'.  When you have bigger text corpous you will observe more changes

##**4.Converting into numerical form/ Feature Engineering**


Now that we have the key tokens created from the text data, we now need a mechanism to convert the tokens into a **numerical form** because we know for a fact that a Machine Learning algorithm works on numerical data. Text data is generally unstructured and varies in its length.There are many techniques and we will use few in this tutorial:


**Bag of Words**

Bag of words allows to convert the text data into numerical vector form by considering the occurrence of the words in text documents, for example:

Doc 1: The best thing in life is to travel

Doc 2: Travel is the best medicine

Doc 3: One should travel more often

**vocabulary**

The list of unique words appearing in all the documents is known as **vocabulary** . In the above example, we have a total of 13 unique words appearing that are part of the vocab as follow.

[The, Best, thing, in, life, is, to, travel, medicine, one, should, more,often]

Any of these three documents can be represented by this vector of fixed size 13 using a Boolean value (1 or 0):

**Doc 1:**
The Best thing in life is to travel medicine one should more often

1   1    1     1  1    1  1  1      0        0    0     0    0

**Doc 2:**

The Best thing in life is to travel medicine one should more often

0   1    0     0   0    0  0   1      1       0    0     0    0


**Doc 3:**

The Best thing in life is to travel medicine one should more often

0   0    0     0  0    0   0   1      0        0    1     1    1

Bag of words does not consider the order of words in the document and the semantic meaning of the word and hence is the most baseline method to represent the text data in numerical form.
There are other ways by which we can convert the textual data into numerical form, which are covered in the following section.
We will use PySpark to go through each one of these methods.

###**CountVectorizer**

In bag of words, we saw the representation of the occurrence of a word by simply **1 or 0** and did not consider the frequency of the word.

The count vectorizer instead takes the **total count of the tokens** appearing in the particular document. We will use the same text documents that we created earlier during tokenization. We first import CountVectorizer:

In [ ]:
from pyspark.ml.feature import CountVectorizer
count_vec=CountVectorizer(inputCol='lemmatized_words',outputCol='features')
cv_df2=count_vec.fit(refined_df2).transform(refined_df2)
cv_df2.select(['user_id','lemmatized_words','features']).show(4,False)

+-------+----------------------------------------------------------------+---------------------------------------------+
|user_id|lemmatized_words                                                |features                                     |
+-------+----------------------------------------------------------------+---------------------------------------------+
|1      |[really, liked, movie, wathing, movie, unforgetable, experience]|(14,[0,2,3,5,8,12],[2.0,1.0,1.0,1.0,1.0,1.0])|
|2      |[recommend, movie, friend]                                      |(14,[0,4,11],[1.0,1.0,1.0])                  |
|3      |[movie, alright, acting, horrible]                              |(14,[0,1,9,13],[1.0,1.0,1.0,1.0])            |
|4      |[never, watching, movie, ever]                                  |(14,[0,6,7,10],[1.0,1.0,1.0,1.0])            |
+-------+----------------------------------------------------------------+---------------------------------------------+



CountVectorizer builds a vocabulary of all the unique words in the corpus and assigns each word an index. The dimensionality of the feature vectors is equal to the size of the vocabulary, which can be very large for large corpora.
As we can observe, each row is represented as a dense vector. It shows that the vector length/dimension is 14 and the first sentence contains **6** word/values of the **vocabulary** at the **0,2,3,5, 8 and 12th** indexes.

Note correspond to the 0th index count is 2. It should be value for the word 'movie' in the first review as count of 'movie' in this sentence is 2.
You can use vocabulary function to see number of unique words in the vocabulary



In [ ]:
#To validate the vocabulary of the count vectorizer, we can simply use the vocabulary function:
vocabulary=count_vec.fit(refined_df2).vocabulary
vocabulary

['movie',
 'horrible',
 'experience',
 'liked',
 'recommend',
 'really',
 'never',
 'watching',
 'unforgetable',
 'alright',
 'ever',
 'friend',
 'wathing',
 'acting']

Hence, the vocabulary size for the preceding sentences is 11; and if you look at the features carefully, they are like the input feature vector we have been using for Machine Learning in PySpark.

The drawback of using the CountVectorizer method is that it doesn’t consider the **co-occurrences of words in other documents.**

In simple terms, the words appearing often would have larger impact on the feature vector. Hence, another approach to convert text data into numerical form is **TF-IDF** (Term Frequency – Inverse Document Frequency).

###**TF-IDF**

This method tries to normalize the frequency of token occurrence based on other documents. The whole idea is to give more weight to the token if appearing high number of times in the same document but penalize if it is appearing higher number of times in other documents as well. This indicates that the token is common across the corpus and is not as important as its frequency in the current document indicates.

**Term Frequency:** Score based on the frequency of the word in the current document

**Inverse Document Frequency:** Scoring based on the number of documents that contain the current word

Now, we create features based on TF-IDF in PySpark using the same refined df Dataframe:

In [ ]:
from pyspark.ml.feature import HashingTF,IDF
hashing_vec2=HashingTF(inputCol='lemmatized_words',outputCol='tf_features')
hashing_df2=hashing_vec2.transform(refined_df2)
hashing_df2.select(['user_id','lemmatized_words','tf_features']).show(4,False)

+-------+----------------------------------------------------------------+---------------------------------------------------------------------------+
|user_id|lemmatized_words                                                |tf_features                                                                |
+-------+----------------------------------------------------------------+---------------------------------------------------------------------------+
|1      |[really, liked, movie, wathing, movie, unforgetable, experience]|(262144,[55875,97005,99172,210223,229264,259272],[1.0,1.0,1.0,2.0,1.0,1.0])|
|2      |[recommend, movie, friend]                                      |(262144,[68228,74520,210223],[1.0,1.0,1.0])                                |
|3      |[movie, alright, acting, horrible]                              |(262144,[95685,171118,210223,236263],[1.0,1.0,1.0,1.0])                    |
|4      |[never, watching, movie, ever]                                  |(262144,[63139,11367

**HashingTF:** It uses a hashing function to map features to indices in a fixed-size feature space. The number of features is determined by the size of the feature space. This means that the dimensionality of the feature vectors is fixed and independent of the actual vocabulary size.

In [ ]:
tf_idf_vec2=IDF(inputCol='tf_features',outputCol='tf_idf_features')
tf_idf_df2=tf_idf_vec2.fit(hashing_df2).transform(hashing_df2)
tf_idf_df2.select(['user_id','tf_idf_features']).show(4,False)

+-------+------------------------------------------------------------------------------------------------------------------------------------------------------+
|user_id|tf_idf_features                                                                                                                                       |
+-------+------------------------------------------------------------------------------------------------------------------------------------------------------+
|1      |(262144,[55875,97005,99172,210223,229264,259272],[0.9162907318741551,0.9162907318741551,0.9162907318741551,0.0,0.9162907318741551,0.9162907318741551])|
|2      |(262144,[68228,74520,210223],[0.9162907318741551,0.9162907318741551,0.0])                                                                             |
|3      |(262144,[95685,171118,210223,236263],[0.9162907318741551,0.9162907318741551,0.0,0.9162907318741551])                                                  |
|4      |(262144,[63139,113673,203

Now we have required features and we can use machine learning models to do something insightful on the preprocessed text corpous. For example, text classification, clustering and so on.  

#**Text Classification Using Machine Learning**

Now that we understand the steps involved in dealing with text processing and feature vectorization, we can build a text classification model and use it for predictions on text data. The dataset that we are going to use is a sample of the open source movie lens reviews data, and we’re going to predict the sentiment of the given review (positive or negative). Let’s start with reading the text data first and creating a Spark Dataframe:

In [ ]:
 # Read CSV file into a DataFrame
 #download the CSV file came along with this tutorial
 #save in ggogle drive and change path according tou your dierectory structure
 text_df= spark.read.csv('drive/My Drive/Files/Movie_reviews.csv',inferSchema=True, header =True)

**Exploratory Data Analysis (EDA)**

Explore data to understand it by looking it number of rows, columns, data types, etc.  

In [ ]:
#check number of columns and their data type
text_df.printSchema()

root
 |-- Review: string (nullable = true)
 |-- Sentiment: string (nullable = true)



As we can see, the Sentiment column is StringType, and we will need to convert it into an integer or float type going forward:

In [ ]:

# Show the first few rows of the DataFrame
text_df.show(10, False)

+------------------------------------------------------------------------+---------+
|Review                                                                  |Sentiment|
+------------------------------------------------------------------------+---------+
|The Da Vinci Code book is just awesome.                                 |1        |
|this was the first clive cussler i've ever read, but even books like Rel|1        |
|i liked the Da Vinci Code a lot.                                        |1        |
|i liked the Da Vinci Code a lot.                                        |1        |
|I liked the Da Vinci Code but it ultimatly didn't seem to hold it's own.|1        |
|that's not even an exaggeration ) and at midnight we went to Wal-Mart to|1        |
|I loved the Da Vinci Code, but now I want something better and different|1        |
|i thought da vinci code was great, same with kite runner.               |1        |
|The Da Vinci Code is actually a good movie...                   

In [ ]:
print((text_df.count(), len(text_df.columns)))

(7087, 2)


We have close to 7k records, out of which some might not be labeled properly. Hence, we filter only those records that are labeled correctly:

In [ ]:
text_df=text_df.filter(((text_df.Sentiment =='1') | (text_df.Sentiment =='0')))
text_df.count()

6990

Some of the records got filtered out, and we are now left with 6990 records for the analysis. The next step is to validate the number of reviews for each class:

In [ ]:
text_df.select('Sentiment').distinct().count()

2

In [ ]:
#We have no unlabeled tweets in the dataset. The next step is to validate the number of reviews for each class:
text_df.groupBy('Sentiment').count().show()

+---------+-----+
|Sentiment|count|
+---------+-----+
|        0| 3081|
|        1| 3909|
+---------+-----+



We are dealing with a **balanced dataset**  here as both classes have an almost similar number of reviews.

As a next step, we create a new integer-type **Label column** and **drop the original Sentiment column,** which was string type:

In [ ]:
text_df=text_df.withColumn("Label",text_df.Sentiment.cast('float')).drop('Sentiment')
from pyspark.sql.functions import rand
text_df.orderBy(rand()).show(10,False)

+------------------------------------------------------------------------+-----+
|Review                                                                  |Label|
+------------------------------------------------------------------------+-----+
|Combining the opinion / review from Gary and Gin Zen, The Da Vinci Code |0.0  |
|i love being a sentry for mission impossible and a station for bonkers. |1.0  |
|THE DA VINCI CODE is AN AWESOME BOOK....                                |1.0  |
|"I liked the first "" Mission Impossible."                              |1.0  |
|for that ( but since it was my fault she is into Harry Potter...        |1.0  |
|so, we got tickets for mission impossible 3, which turned out to be awes|1.0  |
|These Harry Potter movies really suck.                                  |0.0  |
|Not because I hate Harry Potter, but because I am the type of person tha|0.0  |
|I either LOVE Brokeback Mountain or think it's great that homosexuality |1.0  |
|MI3 mission impossible III 

We include an additional column that captures the length of the review:

In [ ]:
from pyspark.sql.functions import length
from pyspark.sql.functions import rand
text_df=text_df.withColumn('length',length(text_df['Review']))
text_df.orderBy(rand()).show(10,False)

+------------------------------------------------------------------------+-----+------+
|Review                                                                  |Label|length|
+------------------------------------------------------------------------+-----+------+
|Brokeback mountain was beautiful...                                     |1.0  |35    |
|da vinci code was an awesome movie...                                   |1.0  |37    |
|Which is why i said silent hill turned into reality coz i was hella like|1.0  |72    |
|the Da Vinci Code sucked.                                               |0.0  |25    |
|Da Vinci Code sucks.                                                    |0.0  |20    |
|I loved seeing Mission Impossible:                                      |1.0  |34    |
|I am the only person in the world who thought Brokeback Mountain sucked.|0.0  |72    |
|Not because I hate Harry Potter, but because I am the type of person tha|0.0  |72    |
|Love luv lubb the Da Vinci Code

In [ ]:
text_df.groupBy('Label').agg({'Length':'mean'}).show()

+-----+-----------------+
|Label|      avg(Length)|
+-----+-----------------+
|  1.0|47.61882834484523|
|  0.0|50.95845504706264|
+-----+-----------------+



There is no major difference between the average length of the positive and negative review.

**Tokenization and Stopwords Removing**

The next step is to start the **tokenization process and remove stopwords:**

In [ ]:
tokenization=Tokenizer(inputCol='Review',outputCol='tokens')
tokenized_df=tokenization.transform(text_df)
tokenized_df.show(10, False)


+------------------------------------------------------------------------+-----+------+----------------------------------------------------------------------------------------+
|Review                                                                  |Label|length|tokens                                                                                  |
+------------------------------------------------------------------------+-----+------+----------------------------------------------------------------------------------------+
|The Da Vinci Code book is just awesome.                                 |1.0  |39    |[the, da, vinci, code, book, is, just, awesome.]                                        |
|this was the first clive cussler i've ever read, but even books like Rel|1.0  |72    |[this, was, the, first, clive, cussler, i've, ever, read,, but, even, books, like, rel] |
|i liked the Da Vinci Code a lot.                                        |1.0  |32    |[i, liked, the, da, vinci, c

In [ ]:
 #OR use RegexTokenizer to only keep the words and remove the special characters such as ')',etc
 # Remove hashtags and special characters from the review column
 from pyspark.ml.feature import RegexTokenizer
 tokenization=RegexTokenizer(inputCol= 'Review' , outputCol= 'tokens', pattern= '\\W')
 tokenized_df=tokenization.transform(text_df)
tokenized_df.show(10, False)

+------------------------------------------------------------------------+-----+------+-----------------------------------------------------------------------------------------+
|Review                                                                  |Label|length|tokens                                                                                   |
+------------------------------------------------------------------------+-----+------+-----------------------------------------------------------------------------------------+
|The Da Vinci Code book is just awesome.                                 |1.0  |39    |[the, da, vinci, code, book, is, just, awesome]                                          |
|this was the first clive cussler i've ever read, but even books like Rel|1.0  |72    |[this, was, the, first, clive, cussler, i, ve, ever, read, but, even, books, like, rel]  |
|i liked the Da Vinci Code a lot.                                        |1.0  |32    |[i, liked, the, da, vin

In [ ]:
#Stop word removal
from pyspark.ml.feature import StopWordsRemover
stopword_removal=StopWordsRemover(inputCol='tokens',outputCol='refined_tokens')
refined_text_df=stopword_removal.transform(tokenized_df)
refined_text_df.show(10, False)
#note the difeerence between tokens and refined_tokens to understand how stop word removal works

+------------------------------------------------------------------------+-----+------+-----------------------------------------------------------------------------------------+---------------------------------------------------------------+
|Review                                                                  |Label|length|tokens                                                                                   |refined_tokens                                                 |
+------------------------------------------------------------------------+-----+------+-----------------------------------------------------------------------------------------+---------------------------------------------------------------+
|The Da Vinci Code book is just awesome.                                 |1.0  |39    |[the, da, vinci, code, book, is, just, awesome]                                          |[da, vinci, code, book, awesome]                               |
|this was the first clive cussle

Since we are now dealing with tokens only instead of an entire review, it would make more sense to capture a number of tokens in each review rather than using the length of the review. We create another column (token count) that gives the number of tokens in each row.

In [ ]:
from pyspark.sql.functions import udf
from pyspark.sql.types import IntegerType
from pyspark.sql.functions import *

In [ ]:
len_udf = udf(lambda s: len(s), IntegerType())
#This is a user defined function (udf), the lambda function takes a single argument s,
#which is a string. It calculates the length of the string s using the built-in len() function
#return an integer value for length of the string.
refined_text_df = refined_text_df.withColumn("token_count", len_udf(col('refined_tokens')))

In [ ]:
refined_text_df.orderBy(rand()).show(10)

+--------------------+-----+------+--------------------+--------------------+-----------+
|              Review|Label|length|              tokens|      refined_tokens|token_count|
+--------------------+-----+------+--------------------+--------------------+-----------+
|Da Vinci Code = U...|  0.0|    72|[da, vinci, code,...|[da, vinci, code,...|          9|
|i liked the Da Vi...|  1.0|    32|[i, liked, the, d...|[liked, da, vinci...|          5|
|Always knows what...|  0.0|    61|[always, knows, w...|[always, knows, w...|          8|
|Brokeback Mountai...|  1.0|    40|[brokeback, mount...|[brokeback, mount...|          4|
|Brokeback Mountai...|  1.0|    49|[brokeback, mount...|[brokeback, mount...|          5|
|"Anyway, thats wh...|  1.0|    49|[anyway, thats, w...|[anyway, thats, l...|          5|
|Not because I hat...|  0.0|    72|[not, because, i,...|[hate, harry, pot...|          6|
|the people who ar...|  1.0|    67|[the, people, who...|[people, worth, k...|          8|
|Ok brokeb

**Converting into numerical form/ Feature Engineering**

Now that we have the refined tokens after stopword removal, we can use any of the preceding approaches to convert text into numerical features. In this case, we use CountVectorizer for feature vectorization for the Machine Learning model:

In [ ]:
from pyspark.ml.feature import CountVectorizer
count_vec=CountVectorizer(inputCol='refined_tokens',outputCol='features')
cv_text_df=count_vec.fit(refined_text_df).transform(refined_text_df)
cv_text_df.select(['refined_tokens','features','Label']).show(10, False)

+---------------------------------------------------------------+----------------------------------------------------------------------------------------+-----+
|refined_tokens                                                 |features                                                                                |Label|
+---------------------------------------------------------------+----------------------------------------------------------------------------------------+-----+
|[da, vinci, code, book, awesome]                               |(1644,[0,1,2,8,167],[1.0,1.0,1.0,1.0,1.0])                                              |1.0  |
|[first, clive, cussler, ve, ever, read, even, books, like, rel]|(1644,[11,41,42,178,180,206,214,806,990,1434],[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0])|1.0  |
|[liked, da, vinci, code, lot]                                  |(1644,[0,1,2,43,182],[1.0,1.0,1.0,1.0,1.0])                                             |1.0  |
|[liked, da, vinci, code, lot]    

(1644,[0,1,2,8,167],[1.0,1.0,1.0,1.0,1.0])

Vocabulary size/dimension is 1644.
Each row is represented as a vector. It shows that the vector length is 1644 and the first review contains five values at the 0,1,2,8,and 167th indexes.All other values shoud be zero

In [ ]:
#explore vocabulary
vocabulary=count_vec.fit(refined_text_df).vocabulary
vocabulary

['da',
 'vinci',
 'code',
 'harry',
 'brokeback',
 'mountain',
 'potter',
 'love',
 'awesome',
 'mission',
 'impossible',
 'like',
 'movie',
 'hate',
 'sucked',
 'sucks',
 'much',
 'really',
 'movies',
 'know',
 'suck',
 '3',
 'want',
 'loved',
 'think',
 'one',
 'stupid',
 'depressing',
 '2',
 'reading',
 'horrible',
 'fucking',
 'terrible',
 'oh',
 'right',
 'left',
 'ok',
 'felicia',
 'beautiful',
 'went',
 'saw',
 'read',
 'first',
 'liked',
 'tom',
 'absolutely',
 'way',
 'heard',
 'big',
 'time',
 'film',
 'said',
 'going',
 'boring',
 'series',
 'watch',
 'great',
 'people',
 'yeah',
 'man',
 're',
 'got',
 'better',
 'watching',
 'story',
 'things',
 'b',
 'last',
 'friday',
 'gay',
 'person',
 'wait',
 'rocks',
 'says',
 'cool',
 'anyone',
 'excellent',
 'always',
 'knows',
 'anyway',
 'mom',
 'friends',
 'review',
 'worth',
 'daniel',
 'opinion',
 'dad',
 'either',
 'care',
 'thats',
 'never',
 'p',
 'guy',
 'stand',
 'luv',
 'gonna',
 'table',
 'awards',
 'hates',
 'head',
 

In [ ]:
# Now we have a feture vector for each row and ready for the next step to make prediction.
#we only use feature vector and Lable to train any ML model, hence select these two columns from the data frame.
model_text_df=cv_text_df.select(['features','Label'])
model_text_df.show()

+--------------------+-----+
|            features|Label|
+--------------------+-----+
|(1644,[0,1,2,8,16...|  1.0|
|(1644,[11,41,42,1...|  1.0|
|(1644,[0,1,2,43,1...|  1.0|
|(1644,[0,1,2,43,1...|  1.0|
|(1644,[0,1,2,43,2...|  1.0|
|(1644,[39,178,639...|  1.0|
|(1644,[0,1,2,22,2...|  1.0|
|(1644,[0,1,2,56,1...|  1.0|
|(1644,[0,1,2,12,1...|  1.0|
|(1644,[0,1,2,167,...|  1.0|
|(1644,[0,1,2,18,2...|  1.0|
|(1644,[0,1,2,167,...|  1.0|
|(1644,[0,1,2,211,...|  1.0|
|(1644,[0,1,2,17,1...|  1.0|
|(1644,[0,1,2,7],[...|  1.0|
|(1644,[0,1,2,23],...|  1.0|
|(1644,[0,1,2,38,2...|  1.0|
|(1644,[0,1,2,8,16...|  1.0|
|(1644,[0,1,2,200,...|  1.0|
|(1644,[0,1,2,188,...|  1.0|
+--------------------+-----+
only showing top 20 rows



Once we have the feature vector for each row, we can make use of VectorAssembler to create input features for the Machine Learning model:


In [ ]:
from pyspark.ml.feature import VectorAssembler
df_assembler = VectorAssembler(inputCols=['features'],outputCol='features_vec')
model_text_df = df_assembler.transform(model_text_df)
model_text_df.printSchema()

root
 |-- features: vector (nullable = true)
 |-- Label: float (nullable = true)
 |-- features_vec: vector (nullable = true)



In [ ]:
model_text_df.show()

+--------------------+-----+--------------------+
|            features|Label|        features_vec|
+--------------------+-----+--------------------+
|(1644,[0,1,2,8,16...|  1.0|(1644,[0,1,2,8,16...|
|(1644,[11,41,42,1...|  1.0|(1644,[11,41,42,1...|
|(1644,[0,1,2,43,1...|  1.0|(1644,[0,1,2,43,1...|
|(1644,[0,1,2,43,1...|  1.0|(1644,[0,1,2,43,1...|
|(1644,[0,1,2,43,2...|  1.0|(1644,[0,1,2,43,2...|
|(1644,[39,178,639...|  1.0|(1644,[39,178,639...|
|(1644,[0,1,2,22,2...|  1.0|(1644,[0,1,2,22,2...|
|(1644,[0,1,2,56,1...|  1.0|(1644,[0,1,2,56,1...|
|(1644,[0,1,2,12,1...|  1.0|(1644,[0,1,2,12,1...|
|(1644,[0,1,2,167,...|  1.0|(1644,[0,1,2,167,...|
|(1644,[0,1,2,18,2...|  1.0|(1644,[0,1,2,18,2...|
|(1644,[0,1,2,167,...|  1.0|(1644,[0,1,2,167,...|
|(1644,[0,1,2,211,...|  1.0|(1644,[0,1,2,211,...|
|(1644,[0,1,2,17,1...|  1.0|(1644,[0,1,2,17,1...|
|(1644,[0,1,2,7],[...|  1.0|(1644,[0,1,2,7],[...|
|(1644,[0,1,2,23],...|  1.0|(1644,[0,1,2,23],...|
|(1644,[0,1,2,38,2...|  1.0|(1644,[0,1,2,38,2...|


**We can use any of the classification model on this data, but we proceed with training a logistic regression model:**

In [ ]:
from pyspark.ml.classification import LogisticRegression
training_df,test_df=model_text_df.randomSplit([0.75,0.25])

In [ ]:
#To verify the presence of enough records for both classes in train and test sets, we can apply the groupBy function on the Label column:
training_df.groupBy('Label').count().show()

+-----+-----+
|Label|count|
+-----+-----+
|  1.0| 2895|
|  0.0| 2295|
+-----+-----+



In [ ]:
test_df.groupBy('Label').count().show()

+-----+-----+
|Label|count|
+-----+-----+
|  1.0| 1014|
|  0.0|  786|
+-----+-----+



In [ ]:
log_reg=LogisticRegression(featuresCol='features_vec',labelCol='Label').fit(training_df)

After training the model, we evaluate the performance of the model on the **test dataset.**

In [ ]:
results=log_reg.evaluate(test_df).predictions
results.show()

+--------------------+-----+--------------------+--------------------+--------------------+----------+
|            features|Label|        features_vec|       rawPrediction|         probability|prediction|
+--------------------+-----+--------------------+--------------------+--------------------+----------+
|(1644,[0,1,2,7],[...|  1.0|(1644,[0,1,2,7],[...|[-29.409028430767...|[1.68974525038430...|       1.0|
|(1644,[0,1,2,7],[...|  1.0|(1644,[0,1,2,7],[...|[-29.409028430767...|[1.68974525038430...|       1.0|
|(1644,[0,1,2,7],[...|  1.0|(1644,[0,1,2,7],[...|[-29.409028430767...|[1.68974525038430...|       1.0|
|(1644,[0,1,2,7],[...|  1.0|(1644,[0,1,2,7],[...|[-29.409028430767...|[1.68974525038430...|       1.0|
|(1644,[0,1,2,7],[...|  1.0|(1644,[0,1,2,7],[...|[-29.409028430767...|[1.68974525038430...|       1.0|
|(1644,[0,1,2,7],[...|  1.0|(1644,[0,1,2,7],[...|[-29.409028430767...|[1.68974525038430...|       1.0|
|(1644,[0,1,2,7],[...|  1.0|(1644,[0,1,2,7],[...|[-29.409028430767...|[1.

In [ ]:
#Evaluate the model performance using confusion matrix
true_postives = results[(results.Label == 1) & (results.prediction == 1)].count()
true_negatives = results[(results.Label == 0) & (results.prediction == 0)].count()
false_positives = results[(results.Label == 0) & (results.prediction == 1)].count()
false_negatives = results[(results.Label == 1) & (results.prediction == 0)].count()

In [ ]:
print ('TP=', true_postives)
print('TN =', true_negatives)
print('FP =', false_positives)
print('FN =', false_negatives)

TP= 1007
TN = 776
FP = 10
FN = 7


In [ ]:
recall = float(true_postives)/(true_postives + false_negatives)
print(recall)

0.9930966469428008


In [ ]:
precision = float(true_postives) / (true_postives + false_positives)
print(precision)

0.9901671583087512


In [ ]:
accuracy=float((true_postives+true_negatives) /(results.count()))
print(accuracy)

0.9905555555555555


**What is Next?**
We used CountVectorizer to create features for classification model. You can use TF-ID and Observe the difference in results using CountVectorizer and TF-IDF model.

Use different classification models which you have learned earlier and identify the bestt model using different feature engineering techniques such as TF-IDF, CountVectorizer.

**Next Lesson**

In the next session we will continue with other text represenation techniques, such as word2vec

In [ ]:
!jupyter nbconvert --to html "/content/drive/MyDrive/ML&Bigdata/CN7030-Feature_Extraction using_TF-IDF.ipynb"

[NbConvertApp] Converting notebook /content/drive/MyDrive/ML&Bigdata/CN7030-Feature_Extraction using_TF-IDF.ipynb to html
[NbConvertApp] Writing 722895 bytes to /content/drive/MyDrive/ML&Bigdata/CN7030-Feature_Extraction using_TF-IDF.html
